# 0004 — Which chunker should the pipeline use?

**Question.** ADR-0005 fixes the hard part: chunks never cross a page. It does
not say where *inside* a page to cut. This notebook answers that by measuring
nine LangChain splitters over the real OCR corpus.

**Decision rule, fixed before the run:**

> maximise the worst `recall@5` across the verbatim and keyword query regimes,
> subject to `chars_p95 ≤ 1500` (the `multilingual-e5-large` ceiling),
> `table_split ≤ 0.10` and `midword_cut ≤ 0.05`; tie-break on needle tightness,
> then on fewer chunks.

**Ground truth.** A *needle*: a deterministic 150–400 character span of 1–3
sentences from a sampled page. The correct chunk is the one containing it whole.
Two query regimes — the needle itself (upper bound) and five content words from
it (a user who does not quote the document) — plus a distractor regime with five
unrelated corpus words added.

**Nothing here needs a key, a model or a network.** Retrieval is BM25 over each
arm's own index, the sample is seeded, so the numbers re-derive exactly.
Runtime is about 5 minutes for three seeds on 8 vCPU.

Design: `docs/plans/2026-10-23-chunking-evaluation-design.md`.
Result: `../chunking/REPORT.md`, `../chunking/results.json`.

## 0. Setup

In [1]:
import hashlib
import json
import math
import random
import re
import statistics
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path

from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "experiments" / "pyproject.toml").is_file()
)
OCR_DIR = ROOT / "data" / "ocr"
RESULTS = ROOT / "experiments" / "chunking" / "results.json"

SEEDS = [20261023, 7, 42]   # one seed would crown a fragile winner
PER_STRATUM = 80            # pages per (length bucket x table) stratum
MIN_PAGE_CHARS = 120

ocr_documents = len([p for p in OCR_DIR.iterdir() if (p / "document.md").is_file()])
print(f"{ocr_documents} extracted documents under {OCR_DIR.relative_to(ROOT)}")

752 extracted documents under data/ocr


## 1. The corpus

`data/ocr/<sha256>/document.md` is what the pipeline will actually see: page
markers, promoted headings, GFM tables. Chunking the PDFs instead would measure
a corpus that never reaches retrieval.

Two properties drive every result below: pages are **long** (p50 2 271
characters) and the corpus is **table-heavy** (6 204 table blocks). A table row
detached from its header is a wrong number, not a shorter answer.

In [2]:
PAGE_MARKER = re.compile(r"<!-- page (\d+) -->")
HEADING = re.compile(r"^#{1,6}\s")
TABLE_ROW = re.compile(r"^\s*\|")


@dataclass(frozen=True, slots=True)
class Page:
    document_id: str
    page_number: int
    text: str

    @property
    def has_table(self) -> bool:
        return "\n|" in self.text or self.text.lstrip().startswith("|")


@dataclass(frozen=True, slots=True)
class Needle:
    document_id: str
    page_number: int
    span: str
    keywords: str
    distractor_keywords: str


def parse_document(markdown: str, document_id: str) -> list[Page]:
    """Split document.md on its page markers; 1-based, like the PDF viewer."""
    parts = PAGE_MARKER.split(markdown)
    pages = []
    for index in range(1, len(parts), 2):
        text = parts[index + 1].strip()
        if text:
            pages.append(Page(document_id, int(parts[index]), text))
    return pages


def load_pages(ocr_dir: Path) -> list[Page]:
    pages = []
    for directory in sorted(p for p in ocr_dir.iterdir() if p.is_dir()):
        document = directory / "document.md"
        if document.is_file():
            pages.extend(parse_document(document.read_text(encoding="utf-8"), directory.name))
    return [page for page in pages if len(page.text) >= MIN_PAGE_CHARS]


ALL_PAGES = load_pages(OCR_DIR)
lengths = sorted(len(page.text) for page in ALL_PAGES)
quantile = lambda fraction: lengths[int(fraction * (len(lengths) - 1))]
print(
    f"{len(ALL_PAGES):,} pages, {sum(lengths):,} characters\n"
    f"p50 {quantile(0.50):,}  p90 {quantile(0.90):,}  p95 {quantile(0.95):,}  "
    f"max {lengths[-1]:,}"
)
print(f"pages with a table: {sum(1 for p in ALL_PAGES if p.has_table):,}")

10,714 pages, 28,783,927 characters
p50 2,379  p90 4,995  p95 6,594  max 21,570
pages with a table: 3,532


### Sampling and needles

A uniform sample would be swamped by medium pages and would rarely see a table,
so pages are stratified by length bucket × presence of a table. Each needle must
occur **exactly once** in its page, which makes "the correct chunk contains the
needle" an exact substring test with no offset bookkeeping — a splitter that
strips whitespace cannot drift the label.

In [3]:
ABBREVIATIONS = (
    "art", "al", "cf", "etc", "m", "mm", "me", "mme", "mlle", "dr", "pr",
    "av", "bd", "chap", "fig", "no", "n°", "p", "pp", "resp", "réf", "ref",
    "vol", "t", "éd", "ed", "ex", "env", "st", "ste", "hab", "apr",
)
_TOKEN = re.compile(r"[0-9a-zàâäçéèêëîïôöùûüÿœæ°%]+")
STOPWORDS = frozenset(
    """
    le la les un une des du de d au aux à et ou mais donc or ni car que qui
    quoi dont où ce cet cette ces son sa ses leur leurs il elle ils elles
    nous vous je tu on se s ne pas plus moins très sur sous dans par pour
    avec sans entre chez est sont était étaient être avoir a ont avait
    comme aussi ainsi lors après avant pendant depuis tout tous toute toutes
    même si non oui
    """.split()
)


def _case_insensitive(literal: str) -> str:
    return "".join(
        f"[{char.upper()}{char.lower()}]" if char.isalpha() else re.escape(char)
        for char in literal
    )


SENTENCE_BOUNDARY = (
    "".join(rf"(?<!\b{_case_insensitive(abbrev)}\.)" for abbrev in sorted(set(ABBREVIATIONS)))
    + r"(?<![A-ZÉÈÀÂÎÔÛÇ]\.)"
    + r"(?<=[.!?])\s+"
)
SENTENCE_END = re.compile(SENTENCE_BOUNDARY)


def sentence_spans(text: str) -> list[tuple[int, int]]:
    """Offsets of French sentences; a period after Art./M./n° does not count."""
    spans, cursor = [], 0
    for match in SENTENCE_END.finditer(text):
        if match.start() > cursor:
            spans.append((cursor, match.start()))
        cursor = match.end()
    if cursor < len(text):
        spans.append((cursor, len(text)))
    trimmed = []
    for start, end in spans:
        while start < end and text[start].isspace():
            start += 1
        while end > start and text[end - 1].isspace():
            end -= 1
        if end > start:
            trimmed.append((start, end))
    return trimmed


def bucket(length: int) -> str:
    if length < 800:
        return "short"
    if length < 2500:
        return "medium"
    if length < 6000:
        return "long"
    return "very-long"


def sample_pages(pages: list[Page], *, seed: int, per_stratum: int = PER_STRATUM) -> list[Page]:
    rng = random.Random(seed)
    strata: dict[tuple[str, bool], list[Page]] = {}
    for page in pages:
        strata.setdefault((bucket(len(page.text)), page.has_table), []).append(page)
    chosen = []
    for key in sorted(strata, key=lambda k: (k[0], k[1])):
        pool = strata[key]
        rng.shuffle(pool)
        chosen.extend(pool[:per_stratum])
    return sorted(chosen, key=lambda p: (p.document_id, p.page_number))


def keywords_of(text: str, count: int, rng: random.Random) -> list[str]:
    seen: dict[str, None] = {}
    for token in _TOKEN.findall(text.lower()):
        if len(token) > 3 and token not in STOPWORDS:
            seen.setdefault(token, None)
    pool = list(seen)
    rng.shuffle(pool)
    pool.sort(key=len, reverse=True)
    return pool[:count]


def needle_from_page(page: Page, rng, distractor_pool, min_chars=150, max_chars=400) -> Needle | None:
    spans = sentence_spans(page.text)
    if len(spans) < 2:
        return None
    starts = list(range(len(spans)))
    rng.shuffle(starts)
    for start in starts:
        for take in range(1, 5):
            if start + take > len(spans):
                break
            span = page.text[spans[start][0] : spans[start + take - 1][1]].strip()
            if len(span) < min_chars:
                continue
            if len(span) > max_chars:
                break
            if HEADING.match(span) or TABLE_ROW.match(span):
                break
            if page.text.count(span) != 1:
                break
            return Needle(
                page.document_id,
                page.page_number,
                span,
                " ".join(keywords_of(span, 5, rng)),
                " ".join([*keywords_of(span, 5, rng), *rng.sample(distractor_pool, 5)]),
            )
    return None


def build_needles(pages: list[Page], *, seed: int) -> list[Needle]:
    rng = random.Random(seed)
    distractor_pool = sorted(
        {
            token
            for page in pages
            for token in keywords_of(page.text, 12, random.Random(page.text[:40]))
        }
    )
    needles = []
    for page in pages:
        needle = needle_from_page(page, rng, distractor_pool)
        if needle is not None:
            needles.append(needle)
    return needles


SAMPLE = sample_pages(ALL_PAGES, seed=SEEDS[0])
NEEDLES = build_needles(SAMPLE, seed=SEEDS[0])
print(f"sample: {len(SAMPLE)} pages -> {len(NEEDLES)} needles")
for needle in NEEDLES[:2]:
    print(f"\n  [{needle.document_id[:8]} p{needle.page_number}] {needle.span[:120]}…")
    print(f"  keywords: {needle.keywords}")

sample: 627 pages -> 405 needles

  [008f6f14 p5] L'indice des prix à la consommation est un indice de Laspeyres (période de base août 2004 avec pondération de 1986-1987)…
  keywords: consommation pondération territoire laspeyres ensemble

  [008f6f14 p9] Tableau 1.3

<table><tr><td></td><td>4e trimestre 2009</td><td>1er trimestre 2010</td><td>2e trimestre 2010</td><td>3e t…
  keywords: trimestre tableau table 2009 juil


## 2. The candidate chunkers

Every arm is a LangChain splitter, or a splitter built on the same sentence layer
as one — benchmarking anything else would not transfer to `chunk.py`.

One global choice: `keep_separator="end"`. LangChain's default (`True`) attaches
a separator to the *following* piece, which on French prose moves the
sentence-final period to the start of the next chunk. Verified before the run;
not a benchmark variable.

In [4]:
FRENCH_SEPARATORS = ["\n\n", "\n", SENTENCE_BOUNDARY, " ", ""]
# The reference implementation's list: splits after "Art.", "M." — and after the
# space in "28,3 %" when a long paragraph forces sentence-level splitting.
NAIVE_FRENCH_SEPARATORS = ["\n\n", "\n", ". ", "; ", ", ", " ", ""]


@dataclass(frozen=True, slots=True)
class Arm:
    name: str
    split: callable
    note: str


def clean(pieces):
    return [piece.strip() for piece in pieces if piece.strip()]


def recursive(chunk_size, chunk_overlap, separators=None, *, is_regex=False):
    kwargs = {
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "keep_separator": "end",
        "is_separator_regex": is_regex,
    }
    if separators is not None:
        kwargs["separators"] = separators
    return RecursiveCharacterTextSplitter(**kwargs)


def fixed_arm(chunk_size):
    splitter = CharacterTextSplitter(
        separator="", chunk_size=chunk_size, chunk_overlap=0, keep_separator="end"
    )
    return lambda text: clean(splitter.split_text(text))


def recursive_arm(chunk_size, chunk_overlap, separators=None, *, is_regex=False):
    splitter = recursive(chunk_size, chunk_overlap, separators, is_regex=is_regex)
    return lambda text: clean(splitter.split_text(text))


def pack(blocks, limit):
    """Greedily join blocks that fit — without this, header chunking fragments."""
    packed, current = [], ""
    for block in blocks:
        if not current:
            current = block
        elif len(current) + 2 + len(block) <= limit:
            current = f"{current}\n\n{block}"
        else:
            packed.append(current)
            current = block
    if current:
        packed.append(current)
    return packed


def markdown_arm(chunk_size, chunk_overlap):
    headers = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("##", "h2"), ("###", "h3")], strip_headers=False
    )
    size_cap = recursive(chunk_size, chunk_overlap, is_regex=True)

    def split(text):
        blocks = []
        for section in headers.split_text(text):
            body = section.page_content.strip()
            if body:
                blocks.extend([body] if len(body) <= chunk_size else size_cap.split_text(body))
        return clean(pack(blocks, chunk_size))

    return split


HASH_DIMENSION = 256


def hash_vector(text: str) -> list[float]:
    vector = [0.0] * HASH_DIMENSION
    for token in re.findall(r"[0-9a-zàâäçéèêëîïôöùûüÿœæ]+", text.lower()):
        digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
        vector[int.from_bytes(digest[:4], "big") % HASH_DIMENSION] += 1.0 if digest[4] & 1 else -1.0
    return vector


def cosine(left, right):
    dot = sum(a * b for a, b in zip(left, right, strict=True))
    norm = math.sqrt(sum(a * a for a in left)) * math.sqrt(sum(b * b for b in right))
    return dot / norm if norm else 0.0


def semantic_shift_arm(max_chars):
    """TextTiling in one pass: cut where adjacent-sentence similarity drops.

    Lexical, not neural — the cheap honest proxy for a neural semantic chunker.
    """
    fallback = recursive(max_chars, 0)

    def split(text):
        spans = sentence_spans(text)
        if len(spans) < 4:
            return clean(fallback.split_text(text))
        vectors = [hash_vector(text[start:end]) for start, end in spans]
        similarities = [cosine(vectors[i], vectors[i + 1]) for i in range(len(vectors) - 1)]
        mean = sum(similarities) / len(similarities)
        variance = sum((value - mean) ** 2 for value in similarities) / len(similarities)
        threshold = mean - math.sqrt(variance)

        chunks, current, length = [], [], 0
        for index, (start, end) in enumerate(spans):
            sentence = text[start:end]
            shifted = index > 0 and similarities[index - 1] < threshold
            overflow = current and length + len(sentence) + 1 > max_chars
            if current and (overflow or (shifted and length >= max_chars // 3)):
                chunks.append(" ".join(current))
                current, length = [], 0
            if not current and len(sentence) > max_chars:
                chunks.extend(clean(fallback.split_text(sentence)))
                continue
            current.append(sentence)
            length += len(sentence) + 1
        if current:
            chunks.append(" ".join(current))
        if len(chunks) > 1 and len(chunks[-1]) < max_chars // 5:
            chunks[-2] = f"{chunks[-2]} {chunks[-1]}"
            chunks.pop()
        return clean(chunks)

    return split


def arms():
    return [
        Arm("fixed-1000", fixed_arm(1000), "control: no boundary respect, no overlap"),
        Arm("recursive-600-100", recursive_arm(600, 100), "small passages for fact lookup"),
        Arm("recursive-1000-0", recursive_arm(1000, 0), "no-overlap recursive, the Chroma result"),
        Arm("recursive-1000-150", recursive_arm(1000, 150), "current Settings default, default separators"),
        Arm("recursive-1500-0", recursive_arm(1500, 0), "context-heavy passages"),
        Arm("recursive-fr-1000-150", recursive_arm(1000, 150, FRENCH_SEPARATORS, is_regex=True),
            "abbreviation-safe French boundaries"),
        Arm("recursive-naivefr-1000-150", recursive_arm(1000, 150, NAIVE_FRENCH_SEPARATORS),
            "reference implementation: splits after Art. / M. / 28,3"),
        Arm("markdown-1000-150", markdown_arm(1000, 150), "structure-aware: ## / ### then size cap"),
        Arm("semantic-shift-1000", semantic_shift_arm(1000), "lexical-shift semantic proxy, no model call"),
    ]


for arm in arms():
    pieces = arm.split(SAMPLE[0].text)
    print(f"{arm.name:<28} {len(pieces)} chunk(s)  first={pieces[0][:60]!r}")

fixed-1000                   3 chunk(s)  first='## Avertissement (suite)\n\n## Prix et autres indicateurs écon'
recursive-600-100            7 chunk(s)  first='## Avertissement (suite)\n\n## Prix et autres indicateurs écon'
recursive-1000-0             4 chunk(s)  first='## Avertissement (suite)\n\n## Prix et autres indicateurs écon'
recursive-1000-150           4 chunk(s)  first='## Avertissement (suite)\n\n## Prix et autres indicateurs écon'
recursive-1500-0             2 chunk(s)  first='## Avertissement (suite)\n\n## Prix et autres indicateurs écon'
recursive-fr-1000-150        4 chunk(s)  first='## Avertissement (suite)\n\n## Prix et autres indicateurs écon'
recursive-naivefr-1000-150   4 chunk(s)  first='## Avertissement (suite)\n\n## Prix et autres indicateurs écon'
markdown-1000-150            3 chunk(s)  first='## Avertissement (suite)\n\n## Prix et autres indicateurs écon'
semantic-shift-1000          3 chunk(s)  first='## Avertissement (suite)\n\n## Prix et autres indicateur

## 3. Metrics

Retrieval is BM25 with inverted postings, so a query touches only the documents
that contain its terms. It is the embedding-free stand-in for the production
retriever: what it measures faithfully is whether a passage is *findable at
all*, which is what the chunker controls.

The shape metrics catch what recall alone rewards: `fixed` windows score well on
recall and cut 76 % of their seams mid-word.

In [5]:
class BM25:
    def __init__(self, texts: list[str], *, k1: float = 1.5, b: float = 0.75) -> None:
        self._k1, self._b = k1, b
        tokenize = lambda text: _TOKEN.findall(text.lower())
        self._lengths = [len(tokens) for tokens in map(tokenize, texts)]
        self._avg = (sum(self._lengths) or 1) / max(len(texts), 1)
        self._postings: dict[str, list[tuple[int, int]]] = defaultdict(list)
        for index, text in enumerate(texts):
            counts: dict[str, int] = defaultdict(int)
            for token in tokenize(text):
                counts[token] += 1
            for token, count in counts.items():
                self._postings[token].append((index, count))
        total = max(len(texts), 1)
        self._idf = {
            token: math.log(1 + (total - len(postings) + 0.5) / (len(postings) + 0.5))
            for token, postings in self._postings.items()
        }

    def search(self, query: str, *, top_k: int = 5) -> list[int]:
        scores: dict[int, float] = defaultdict(float)
        for token in set(_TOKEN.findall(query.lower())):
            for index, frequency in self._postings.get(token, ()):
                norm = 1 - self._b + self._b * self._lengths[index] / (self._avg or 1)
                scores[index] += self._idf[token] * frequency * (self._k1 + 1) / (
                    frequency + self._k1 * norm
                )
        return [index for index, _ in sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))[:top_k]]

In [6]:
TABLE_BLOCK = re.compile(r"(?:^\s*\|.*(?:\n|$))+", re.MULTILINE)
TABLE_BLOCK_MAX = 900  # a 2 000-char table cannot survive any 1 000-char chunker
ABBREV_END = re.compile(
    r"\b(?:art|al|cf|etc|m|mm|me|mme|mlle|dr|pr|av|bd|chap|fig|n°|no|p|pp"
    r"|resp|réf|ref|vol|éd|ed|ex|env|st|ste|hab)\.?\s*$",
    re.IGNORECASE,
)
LOWERCASE_START = re.compile(r"^[a-zàâäçéèêëîïôöùûüÿœæ]")


def percentile(values, fraction):
    if not values:
        return 0
    ordered = sorted(values)
    return ordered[min(int(fraction * (len(ordered) - 1)), len(ordered) - 1)]


def chunk_pages(arm: Arm, pages: list[Page]):
    texts, by_page = [], {}
    for page in pages:
        indices = []
        for piece in arm.split(page.text):
            indices.append(len(texts))
            texts.append(piece)
        by_page[(page.document_id, page.page_number)] = indices
    return texts, by_page


def shape_metrics(texts, by_page, pages) -> dict:
    lengths = [len(text) for text in texts]
    page_text = {(page.document_id, page.page_number): page.text for page in pages}
    abbrev = orphans = lowercase = midword = pairs = seams = 0
    for key, indices in by_page.items():
        for position, index in enumerate(indices):
            text = texts[index]
            abbrev += bool(ABBREV_END.search(text))
            lowercase += bool(LOWERCASE_START.match(text))
            orphans += bool(re.match(r"^#{1,6}\s", text) and len(text) < 80)
            if position:
                pairs += 1
                previous = texts[indices[position - 1]]
                seams += bool(previous and text and previous[-1].isalpha() and text[0].isalpha())
                # A real mid-word cut leaves the two slices contiguous in the page.
                tail, head = previous[-24:], text[:24]
                midword += len(tail) + len(head) >= 8 and (tail + head) in page_text[key]

    table_blocks = split_tables = 0
    for page in pages:
        page_chunks = [texts[i] for i in by_page[(page.document_id, page.page_number)]]
        for block in TABLE_BLOCK.findall(page.text):
            block = block.strip()
            if 40 < len(block) <= TABLE_BLOCK_MAX:
                table_blocks += 1
                split_tables += not any(block in chunk for chunk in page_chunks)

    count = max(len(texts), 1)
    return {
        "chunk_count": len(texts),
        "chars_p50": percentile(lengths, 0.50),
        "chars_p95": percentile(lengths, 0.95),
        "chars_max": max(lengths) if lengths else 0,
        "chars_mean": round(statistics.fmean(lengths), 1) if lengths else 0.0,
        "chunks_per_page": round(len(texts) / max(len(pages), 1), 2),
        "abbrev_cut": round(abbrev / count, 4),
        "mid_sentence_start": round(lowercase / count, 4),
        "midword_cut": round(midword / max(pairs, 1), 4),
        "letter_seam": round(seams / max(pairs, 1), 4),
        "table_split": round(split_tables / max(table_blocks, 1), 4),
        "table_blocks": table_blocks,
        "heading_orphan": round(orphans / count, 4),
    }

In [7]:
REGIMES = {"verbatim": "span", "keywords": "keywords", "distractor": "distractor_keywords"}


def retrieval_metrics(texts, by_page, needles, *, top_k: int = 5):
    index = BM25(texts)
    results = {}
    for regime, attribute in REGIMES.items():
        hit_at_1 = hit_at_k = page_hits = 0
        reciprocal = tightness = 0.0
        boundary_loss = 0
        for needle in needles:
            page_indices = by_page[(needle.document_id, needle.page_number)]
            if not any(needle.span in texts[i] for i in page_indices):
                boundary_loss += 1

            ranked = index.search(getattr(needle, attribute), top_k=top_k)
            rank = next(
                (position for position, i in enumerate(ranked) if needle.span in texts[i]), None
            )
            if rank is not None:
                hit_at_k += 1
                reciprocal += 1 / (rank + 1)
                tightness += len(needle.span) / max(len(texts[ranked[rank]]), 1)
                hit_at_1 += rank == 0
            # The citation contract needs the right *page* (`#page=N`), even
            # when the exact passage was split across two chunks.
            page_hits += any(i in page_indices for i in ranked)

        scored = max(len(needles), 1)
        results[regime] = {
            "recall@1": round(hit_at_1 / scored, 4),
            "recall@5": round(hit_at_k / scored, 4),
            "page_recall@5": round(page_hits / scored, 4),
            "mrr@5": round(reciprocal / scored, 4),
            "needle_tightness": round(tightness / max(hit_at_k, 1), 4),
            "boundary_loss": round(boundary_loss / scored, 4),
            "queries": len(needles),
        }
    return results

## 4. Run every arm on three independent samples

One seed can crown a fragile winner. Three seeds are the minimum that shows
whether a ranking survives resampling, so the table reports the mean and the
min–max of the deciding metric.

In [8]:
def evaluate(pages, needles):
    rows = []
    for arm in arms():
        texts, by_page = chunk_pages(arm, pages)
        rows.append(
            {
                "name": arm.name,
                "note": arm.note,
                "shape": shape_metrics(texts, by_page, pages),
                "regimes": retrieval_metrics(texts, by_page, needles),
            }
        )
    return rows


def leaves(row, prefix=""):
    flat = {}
    for key, value in row.items():
        if key in ("name", "note"):
            continue
        if isinstance(value, dict):
            flat.update(leaves(value, f"{prefix}{key}."))
        elif isinstance(value, (int, float)):
            flat[f"{prefix}{key}"] = float(value)
    return flat


def set_path(row, path, value):
    keys = path.split(".")
    node = row
    for key in keys[:-1]:
        node = node[key]
    node[keys[-1]] = value


def aggregate(per_seed):
    by_arm = {}
    for rows in per_seed:
        for row in rows:
            by_arm.setdefault(row["name"], []).append(row)
    means, spread = [], {}
    for name, runs in by_arm.items():
        flats = [leaves(run) for run in runs]
        mean_row = json.loads(json.dumps(runs[0]))
        for path in flats[0]:
            values = [flat[path] for flat in flats]
            set_path(mean_row, path, round(sum(values) / len(values), 4))
            spread[f"{name}::{path}"] = {"min": min(values), "max": max(values)}
        means.append(mean_row)
    return means, spread


per_seed, sampled, needle_count = [], 0, 0
for seed in SEEDS:
    pages = sample_pages(ALL_PAGES, seed=seed)
    needles = build_needles(pages, seed=seed)
    sampled += len(pages)
    needle_count += len(needles)
    print(f"seed {seed}: {len(pages)} pages, {len(needles)} needles", flush=True)
    per_seed.append(evaluate(pages, needles))

rows, spread = aggregate(per_seed)
print("\ndone")

seed 20261023: 627 pages, 405 needles


seed 7: 627 pages, 395 needles


seed 42: 627 pages, 417 needles



done


In [9]:
header = (
    "| arm | chunks | p50 | p95 | rec@5v | rec@5k (min–max) | rec@5d | page@5 "
    "| tight | b-loss | seam | mid-s | abbrev | table |"
)
print(header)
print("|" + "---|" * 14)
for row in rows:
    shape, verbatim, keywords = row["shape"], row["regimes"]["verbatim"], row["regimes"]["keywords"]
    distractor = row["regimes"]["distractor"]
    bounds = spread[f"{row['name']}::regimes.keywords.recall@5"]
    print(
        f"| `{row['name']}` | {shape['chunk_count']:.0f} | {shape['chars_p50']:.0f} "
        f"| {shape['chars_p95']:.0f} | {verbatim['recall@5']:.3f} | {keywords['recall@5']:.3f} "
        f"({bounds['min']:.3f}–{bounds['max']:.3f}) | {distractor['recall@5']:.3f} "
        f"| {keywords['page_recall@5']:.3f} | {keywords['needle_tightness']:.3f} "
        f"| {max(verbatim['boundary_loss'], keywords['boundary_loss'], distractor['boundary_loss']):.3f} "
        f"| {shape['midword_cut']:.3f} | {shape['mid_sentence_start']:.3f} "
        f"| {shape['abbrev_cut']:.3f} | {shape['table_split']:.3f} |"
    )

| arm | chunks | p50 | p95 | rec@5v | rec@5k (min–max) | rec@5d | page@5 | tight | b-loss | seam | mid-s | abbrev | table |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| `fixed-1000` | 2467 | 1000 | 1000 | 0.882 | 0.791 (0.753–0.820) | 0.766 | 0.897 | 0.282 | 0.116 | 0.759 | 0.296 | 0.008 | 0.434 |
| `recursive-600-100` | 5476 | 470 | 598 | 0.616 | 0.567 (0.552–0.595) | 0.563 | 0.871 | 0.480 | 0.383 | 0.001 | 0.233 | 0.001 | 0.548 |
| `recursive-1000-0` | 3156 | 806 | 996 | 0.677 | 0.631 (0.620–0.638) | 0.628 | 0.862 | 0.350 | 0.322 | 0.021 | 0.181 | 0.001 | 0.004 |
| `recursive-1000-150` | 3224 | 820 | 996 | 0.705 | 0.658 (0.651–0.669) | 0.654 | 0.866 | 0.349 | 0.294 | 0.001 | 0.204 | 0.000 | 0.004 |
| `recursive-1500-0` | 2276 | 1015 | 1494 | 0.738 | 0.697 (0.691–0.701) | 0.692 | 0.843 | 0.278 | 0.261 | 0.015 | 0.149 | 0.000 | 0.004 |
| `recursive-fr-1000-150` | 3323 | 797 | 995 | 0.711 | 0.660 (0.652–0.671) | 0.657 | 0.896 | 0.365 | 0.288 | 0.001 | 0.177 | 0.001 | 0.00

### The rule, applied

Gates are defects a chunker must not have: blow the embedder ceiling, cut
mid-word, or destroy a table that would have fitted. `boundary_loss` is reported
rather than gated — a 400-character needle inside a 1 000-character chunker
crosses a cut about 30 % of the time by arithmetic alone.

In [10]:
MAX_P95, MAX_TABLE, MAX_SEAM = 1500, 0.10, 0.05


def recall(row):
    return min(row["regimes"][r]["recall@5"] for r in ("verbatim", "keywords"))


eligible = [
    row
    for row in rows
    if row["shape"]["chars_p95"] <= MAX_P95
    and row["shape"]["table_split"] <= MAX_TABLE
    and row["shape"]["midword_cut"] <= MAX_SEAM
]
pool = eligible or rows
chosen = max(
    pool,
    key=lambda row: (recall(row), row["regimes"]["keywords"]["needle_tightness"],
                     -row["shape"]["chunk_count"]),
)
runner_up = max((row for row in pool if row is not chosen), key=recall, default=None)

verdict = {
    "rule": (
        "max min(recall@5 over verbatim,keywords) subject to chars_p95<=1500, "
        "table_split<=0.10, midword_cut<=0.05; tie-break tightness then fewer chunks"
    ),
    "eligible": [row["name"] for row in eligible],
    "chosen": chosen["name"],
    "chosen_recall@5": round(recall(chosen), 4),
    "runner_up": runner_up["name"] if runner_up else None,
    "runner_up_recall@5": round(recall(runner_up), 4) if runner_up else None,
    "constrained_out": [row["name"] for row in rows if row not in eligible],
}
print(json.dumps(verdict, indent=2, ensure_ascii=False))

{
  "rule": "max min(recall@5 over verbatim,keywords) subject to chars_p95<=1500, table_split<=0.10, midword_cut<=0.05; tie-break tightness then fewer chunks",
  "eligible": [
    "recursive-1000-0",
    "recursive-1000-150",
    "recursive-1500-0",
    "recursive-fr-1000-150",
    "recursive-naivefr-1000-150",
    "semantic-shift-1000"
  ],
  "chosen": "semantic-shift-1000",
  "chosen_recall@5": 0.7978,
  "runner_up": "recursive-1500-0",
  "runner_up_recall@5": 0.6967,
  "constrained_out": [
    "fixed-1000",
    "recursive-600-100",
    "markdown-1000-150"
  ]
}


## 5. Write the artifact

`results.json` is committed so the report's numbers can be re-derived without
re-running the notebook, and so a later change to the chunker has something to
regress against.

In [11]:
payload = {
    "seeds": SEEDS,
    "corpus": {
        "documents": len({page.document_id for page in ALL_PAGES}),
        "pages": len(ALL_PAGES),
        "chars": sum(len(page.text) for page in ALL_PAGES),
        "sampled_pages_mean": round(sampled / len(SEEDS)),
        "needles_mean": round(needle_count / len(SEEDS)),
    },
    "arms": rows,
    "spread": spread,
    "verdict": verdict,
}
RESULTS.parent.mkdir(parents=True, exist_ok=True)
RESULTS.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"wrote {RESULTS.relative_to(ROOT)}")

wrote experiments/chunking/results.json


## 6. What it says

**`semantic-shift-1000` wins**, and it wins on metrics it was not tuned on.

| | semantic-shift | recursive 1500/0 | recursive 1000/150 |
|---|---|---|---|
| recall@5, keywords | **0.798** (0.780–0.811) | 0.697 | 0.658 |
| recall@5, verbatim | **0.871** | 0.738 | 0.705 |
| page recall@5 | **0.912** | 0.843 | 0.866 |
| needle tightness | **0.450** | 0.278 | 0.349 |
| boundary loss | **0.128** | 0.261 | 0.294 |
| chunks | 3 139 | 2 276 | 3 224 |
| p95 characters | 997 | 1 494 | 996 |
| tables split | 4,6 % | 0,4 % | 0,4 % |

Four things worth carrying into the implementation:

1. **Recall alone is a trap.** `fixed-1000` scores 0.791 on keywords — better
   than every recursive arm — while cutting 76 % of its seams mid-word and
   destroying 43 % of the tables that would have fitted. Any future gate that
   only watches recall will pick it.
2. **Bigger windows buy recall by diluting the passage.** recursive 1000 → 1500
   gains 4 points of recall and loses 7 points of tightness, and lands on the
   embedder ceiling (p95 1 494). Cutting where the topic shifts buys both.
3. **"Structure-aware" is not automatically better.** `MarkdownHeaderTextSplitter`
   rewrites blank lines into `"  \n"`, which removes the paragraph separators the
   size-cap pass needs; it falls through to single-newline splits and cuts 23,5 %
   of tables. It scored 0.592, below plain recursive.
4. **French separators are free but nearly irrelevant at 1 000 characters.**
   `recursive-fr` 0.660 vs `recursive-naivefr` 0.659 — almost every cut lands on
   a *paragraph* boundary, not a sentence one. Keep the abbreviation-safe
   separator anyway: it cuts abbreviation-ending chunks 5× less often, and the
   naive list's `keep_separator=False` drops the sentence-final period.

**Caveats.** BM25 is not `multilingual-e5-large`; five content words are not a
French interrogative; and the winning arm and the retriever share a lexical
bias — the verbatim regime and `boundary_loss` are independent of it and favour
the same arm, but the choice must be re-checked against the real embedder once
an index exists. See `../chunking/REPORT.md`.